In [ ]:
import sys
from pathlib import Path

projeto_raiz = Path("~/Documentos/ESPECT_CONV").expanduser().resolve()

if str(projeto_raiz) not in sys.path:
    sys.path.append(str(projeto_raiz))

# ==========================================================
# CONFIGURAÇÕES DE CAMINHO & IMPORTS LOCAIS
# ==========================================================

from pipeline.stage_20_load_and_epoching.stage_20_load_and_epoching import load_and_epoching

from pipeline.stage_30_psd_array_assembly.channels_2D_distribution import CENTRALIZED_ONE_BY_ONE_2D_DISTRIBUTION
from pipeline.stage_30_psd_array_assembly.stage_30_psd_array_assembly import psd_array_assembly
from pipeline.stage_40_obtain_indices.obtain_indices import obtain_indices
from pipeline.stage_50_execute_branch.execute_branch import execute_branch
from pipeline.stage_60_execute_fusion.execute_fusion import execute_fusion


CONDITION = "INNER_SPEECH"


# Stage 20: Load and Epoching

In [ ]:

EPOCH_START_SECONDS = 1.0
EPOCH_END_SECONDS = 3.5

# ======================================================
# CONDITION OPTIONS
# ======================================================
#
# "PRONOUNCED_SPEECH"
#
# "INNER_SPEECH"
#
# "VISUALIZED_CONDITION"
#
# ======================================================



load_and_epoching(
        condition=CONDITION,
        crop_start_time=EPOCH_START_SECONDS,
        crop_end_time=EPOCH_END_SECONDS
)

# Stage 30: PSD Array Assembly

In [ ]:
WINDOW_DURATION = 0.5
STEP_DURATION = 0.25

psd_array_assembly(
        window_duration = WINDOW_DURATION,
        step_duration = STEP_DURATION,
        channels_distribution = CENTRALIZED_ONE_BY_ONE_2D_DISTRIBUTION
)

# Stage 40: Obtain Indices

In [ ]:

subjects = [
    f"sub-{i:02d}"
    for i in range(1, 11)
]

N_FOLDS = 6
VALIDATION_SIZE = 0.20
RANDOM_STATE = 42

obtain_indices(
    subjects=subjects,
    N_FOLDS=N_FOLDS,
    VALIDATION_SIZE=VALIDATION_SIZE,
    RANDOM_STATE=RANDOM_STATE,
)

# Stage 50: Execute Branch

In [ ]:

# ======================================================
# BRANCH OPTIONS
# ======================================================
#
# space1_space2_frequency_time
#
# space1_space2_time_frequency
#
# time_frequency_space2_space1
#
# time_frequency_space1_space2
#
# ======================================================

subjects = [
    f"sub-{i:02d}"
    for i in range(4, 11)
]
# subjects = ["sub-01"]

branches = [
    "space1_space2_frequency_time", 
    "space1_space2_time_frequency", 
    "time_frequency_space2_space1", 
    "time_frequency_space1_space2" 
]

for subject in subjects:
    for branch in branches:

        execute_branch(
            branch=branch,
            N_EPOCHS=600,       
            subjects= subject
        )

# Stage 60: Execute Fusion

In [ ]:

# ======================================================
# BRANCH_MODEL_STATE OPTIONS
# ======================================================
#
# minimum_loss_model
#
# maximum_accuracy_model
#
# ======================================================

# ======================================================
# FUSION OPTIONS
# ======================================================
#
# mean_fusion
# concatenation_fusion
# multiplication_fusion
# multiplication_log_softmax_fusion
# weighted_sum_fusion
# scalar_gated_fusion
# class_wise_gated_fusion
#
#
# ======================================================


# subjects = [
#     f"sub-{i:02d}"
#     for i in range(1, 11)
# ]
subjects = ["sub-01"]

branch_mode_states = ["minimum_loss_model", "maximum_accuracy_model"]

# fusions = [
#     "mean_fusion",
#     "concatenation_fusion",
#     "weighted_sum_fusion",
#     "scalar_gated_fusion",
#     "class_wise_gated_fusion"
# ]

fusions = [
    # "mean_fusion",
    # "concatenation_fusion",
    "weighted_sum_fusion",
    # "scalar_gated_fusion",
    # "class_wise_gated_fusion"
]


for subject in subjects:
    for branch_mode_state in branch_mode_states:
        for fusion in fusions:

            execute_fusion(
                fusion= fusion,
                branch_model_state = branch_mode_state,
                subjects=subject,
                condition_folder = CONDITION,
                N_EPOCHS_STAGE_1 = 1200,
                N_EPOCHS_STAGE_2 = 300,
                LEARNING_RATE_STAGE_1 = 0.001,
                LEARNING_RATE_STAGE_2 = 0.0001
            )